# Data preprocessing

In [1]:
%pip install --upgrade pip
%pip install -q langdetect
%pip install numpy pandas

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Import necessary dependencies

In [2]:
import os, re, json, html, unicodedata
import pandas as pd
from collections import Counter

from langdetect import detect, DetectorFactory

## Define Constants

In [3]:
CONFIG = json.load(open("config.json"))

SRC_PATH   = CONFIG["src_path"]
OUTPUT_DIR = CONFIG["output_dir"]
CLEAN_PATH = CONFIG["clean_path"]
DOMAIN_TAGS = CONFIG["domain_tags"]
MIN_BODY_CHARS = 50
MIN_TAG_FREQUENCY = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Fetch the dataset

In [4]:
df = pd.read_csv(SRC_PATH)

df.head(5)

,question_id,title,body_text,code_text,tags,score,view_count,answer_count,is_answered,has_accepted_answer,creation_date,closed_reason,duplicate_of
0,77740646,Make URL Address Text a Clickable URL Using se...,Happy New Year to Everyone.. I have a list of ...,function createPDFDocuments() {\n var sheetD...,google-apps-script|pdf-generation,2,744,1,True,True,2024-01-01T00:01:14+00:00,NaN,NaN
1,77740652,7zip commandline extract excluding single file...,Using 7z.exe (v19.00 x64) from the commandline...,"""C:\Program Files\7-Zip\7z"" x -bb3 -x""Www.WebA...",extract|7zip,0,554,1,True,True,2024-01-01T00:04:07+00:00,NaN,NaN
2,77740661,How do I delete an element from an array and s...,Im building a chrome extension that lets you s...,"$(""ul"").on(""click"", "".removeButton"", function(...",javascript|arrays|google-chrome-extension|goog...,-1,31,0,False,False,2024-01-01T00:08:51+00:00,Duplicate,20279484
3,77740664,Issues with JumpList in C#,I am working on a C# application and trying to...,CS0176: Member 'JumpList.AddToRecent(string)' ...,c#|jump-list,0,115,0,False,False,2024-01-01T00:13:08+00:00,NaN,NaN
4,77740666,Determine whether app launches AOT binary (or ...,I am developing an Android Test App and was wo...,$ adb shell cmd package compile -m speed -f my...,android,1,120,0,False,False,2024-01-01T00:14:37+00:00,NaN,NaN


### Describe dataset

In [5]:
df.describe()

,question_id,score,view_count,answer_count
count,1.315189e+06,1.315189e+06,1.315189e+06,1.315189e+06
mean,7.728323e+07,8.133873e-01,9.861632e+02,1.010976e+00
std,2.019821e+06,2.367227e+01,2.785858e+04,9.118194e-01
min,3.061000e+03,-2.600000e+01,5.000000e+00,0.000000e+00
25%,7.611211e+07,0.000000e+00,9.000000e+01,1.000000e+00
50%,7.726232e+07,0.000000e+00,1.850000e+02,1.000000e+00
75%,7.847925e+07,1.000000e+00,5.230000e+02,1.000000e+00
max,8.001862e+07,8.450000e+03,7.568446e+06,7.000000e+01


#### Shape of dataset

In [6]:
df.shape

(1315189, 13)

### Data types of dataset

In [7]:
print(df.dtypes)

question_id            int64
title                    str
body_text                str
code_text                str
tags                     str
score                  int64
view_count             int64
answer_count           int64
is_answered             bool
has_accepted_answer     bool
creation_date            str
closed_reason            str
duplicate_of             str
dtype: object


#### Check for missing values

In [8]:
print(df.isnull().sum())

question_id                  0
title                        0
body_text                  462
code_text               290507
tags                         2
score                        0
view_count                   0
answer_count                 0
is_answered                  0
has_accepted_answer          0
creation_date                0
closed_reason          1252715
duplicate_of           1274483
dtype: int64


## Type cast

In [9]:
df["question_id"] = df["question_id"].astype(str)
df["creation_date"] = pd.to_datetime(df["creation_date"], utc=True)

print("Datatypes of columns:")
print(df.dtypes)

Datatypes of columns:
question_id                            str
title                                  str
body_text                              str
code_text                              str
tags                                   str
score                                int64
view_count                           int64
answer_count                         int64
is_answered                           bool
has_accepted_answer                   bool
creation_date          datetime64[us, UTC]
closed_reason                          str
duplicate_of                           str
dtype: object


## Drop duplicate question by id if exists

In [10]:
df_count_before = len(df)
df = df.drop_duplicates(subset="question_id").reset_index(drop=True)

print("Removing by question id:")
print(f"Before count: {df_count_before}")
print(f"After count: {len(df)}")
print(f"Total removed by question_id: {df_count_before - len(df)}")

Removing by question id:
Before count: 1315189
After count: 1315189
Total removed by question_id: 0


## Domain filter based on tags related to AI domain


In [11]:
domains = set(DOMAIN_TAGS)

df["tag_list"] = df["tags"].fillna("").str.split("|")

df_count_before = len(df)
df = df[df["tag_list"].map(lambda ts: bool(set(ts) & domains))].reset_index(drop=True)

print("Filter out by tags:")
print(f"Before count: {df_count_before}")
print(f"After count: {len(df)}")
print(f"Total removed by tags: {df_count_before - len(df)}")

counts = Counter(t for ts in df["tag_list"] for t in ts)
print("\nTop 10 tags: " + str(counts.most_common(10)))
df = df.drop(columns=["tag_list"])


Filter out by tags:
Before count: 1315189
After count: 57794
Total removed by tags: 1257395

Top 10 tags: [('python', 33608), ('numpy', 8265), ('matplotlib', 5905), ('pytorch', 5447), ('tensorflow', 4916), ('machine-learning', 4803), ('pandas', 3701), ('opencv', 3470), ('jupyter-notebook', 2988), ('deep-learning', 2809)]


## Drop closed question and duplicate of columns

In [12]:
df_count_before = len(df)
df['closed_reason'] = df['closed_reason'].fillna("")
df = df[df["closed_reason"] == ""].reset_index(drop=True)

# closed_reason and duplicate_of are not used for recommendation
df = df.drop(columns=["closed_reason", "duplicate_of"])

print("Before count:", df_count_before)
print("After count:", len(df))
print(f"Dropped closed questions count: {df_count_before - len(df)}")


Before count: 57794
After count: 55684
Dropped closed questions count: 2110


## Check missing values and removing if any


In [13]:
cols = ["body_text", "code_text"]
df[cols] = df[cols].fillna("")
print(f"filled missing values in: {cols}")
print(df.isnull().sum())

filled missing values in: ['body_text', 'code_text']
question_id            0
title                  0
body_text              0
code_text              0
tags                   0
score                  0
view_count             0
answer_count           0
is_answered            0
has_accepted_answer    0
creation_date          0
dtype: int64


## Text cleaning


In [14]:
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
MARKDOWN_IMAGE_PATTERN = re.compile(r"!\[[^\]]*\]\([^)]*\)")
HTML_TAG_PATTERN = re.compile(r"</?(?:p|div|br|ul|ol|li|h[1-6]|blockquote|table|tr|td|th|a|img|span|strong|em|code)\b[^>]*>", re.I)

def clean_text(raw_text):
    if not isinstance(raw_text, str):
        return ""
    cleaned_text = html.unescape(raw_text)
    cleaned_text = MARKDOWN_IMAGE_PATTERN.sub(" <IMG> ", cleaned_text)
    cleaned_text = URL_PATTERN.sub(" <URL> ", cleaned_text)
    cleaned_text = HTML_TAG_PATTERN.sub(" ", cleaned_text)
    cleaned_text = unicodedata.normalize("NFKC", cleaned_text)
    return " ".join(cleaned_text.split())

for column_name in ["title", "body_text"]:
    df[column_name] = df[column_name].map(clean_text)

df["text_all"] = (df["title"] + " " + df["body_text"]).str.lower()

df[["title", "body_text", "text_all"]].head(5)

,title,body_text,text_all
0,New-AzCognitiveServicesAccount,I've created a subdomain in Azure Cloud Shell ...,new-azcognitiveservicesaccount i've created a ...
1,How do i know if Yolov8 training working corre...,I'm new at programming and for a homework i wa...,how do i know if yolov8 training working corre...
2,"matplotlib, ticks are overlapped when hspace=0","When pushing two plots together, I'd like to h...","matplotlib, ticks are overlapped when hspace=0..."
3,No module named keras error when I use from ke...,"In this code, no module named 'keras' error ha...",no module named keras error when i use from ke...
4,How can I avoid this Numpy ArrayMemoryError wh...,I'm getting a numpy.core._exceptions._ArrayMem...,how can i avoid this numpy arraymemoryerror wh...


## Accept only English language

In [15]:
DetectorFactory.seed = 0

def is_english(row):
    text = (row["title"] + " " + row["body_text"])[:500]
    if not text.strip():
        return True
    try:
        return detect(text) == "en"
    except Exception:
        return True

df_count_before = len(df)
df = df[df.apply(is_english, axis=1)].reset_index(drop=True)

print("Before count:", df_count_before)
print("After count:", len(df))
print("Removed non-English rows:", df_count_before - len(df))

Before count: 55684
After count: 55668
Removed non-English rows: 16


## Quality filters


In [16]:
df_count_before = len(df)
df = df[df["body_text"].str.len() >= MIN_BODY_CHARS]
print(f"dropped short-body questions: {df_count_before - len(df)}")

df_count_before = len(df)
df = df[df["answer_count"] > 0].reset_index(drop=True)
print(f"dropped questions with no answers: {df_count_before - len(df)}")

dropped short-body questions: 94
dropped questions with no answers: 15465


## Drop rare domain tags


In [17]:
def domain_tags_of(tag_string):
    return [tag for tag in tag_string.split("|") if tag in domains]

df_count_before = len(df)
while True:
    tag_counts = Counter(tag for tags in df["tags"] for tag in domain_tags_of(tags))
    rare_tags = {tag for tag, count in tag_counts.items() if count < MIN_TAG_FREQUENCY}
    if not rare_tags:
        break
    df["tags"] = df["tags"].map(lambda tags: "|".join(tag for tag in tags.split("|") if tag not in rare_tags))
    df = df[df["tags"].map(lambda tags: bool(domain_tags_of(tags)))].reset_index(drop=True)

print(f"rare domain tags (< {MIN_TAG_FREQUENCY} questions) removed: {sorted(rare_tags)}")
print(f"dropped questions left with no domain tag: {df_count_before - len(df)}")
print(f"domain tags kept: {len(tag_counts)}")

rare domain tags (< 5 questions) removed: []
dropped questions left with no domain tag: 0
domain tags kept: 179


### Shape of clean dataset

In [18]:
df.shape

(40109, 12)

## Save clean dataset

In [19]:
df.to_csv(CLEAN_PATH, index=False)
print(f"saved clean dataset")

saved clean dataset
